# Gemma 4 Developer Agent package

This competition notebook writes a validated agent ZIP to `/kaggle/working/submission.zip`, with `agent.yaml` at the archive root. It embeds only the selected agent files; it does not use local issue solutions, hidden labels, or answer patches. The competition scoring harness supplies the required Gemma 4 model when it evaluates the ZIP.

Run all cells, then submit this notebook version's generated ZIP.


In [ ]:
from pathlib import Path
import json, os, platform, subprocess

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
COMP = Path('/kaggle/input/competitions/gemma-4-developer-agent')
print('Working directory:', WORK)
print('Python:', platform.python_version())
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True, capture_output=True, timeout=10)
    print('Visible GPUs:', gpu.stdout.strip() or gpu.stderr.strip() or 'none')
except Exception as exc:
    print('GPU inventory unavailable:', type(exc).__name__)
tasks = COMP / 'tasks.jsonl'
if tasks.is_file():
    with tasks.open(encoding='utf-8') as stream:
        count = sum(1 for line in stream if line.strip())
    print('Public task rows:', count)
else:
    print('Public tasks.jsonl is not mounted; ZIP build does not depend on it.')


## Build a clean, root-layout submission archive

The files below are embedded from the selected agent directory. The ZIP writer uses only that explicit file map, so notebook outputs, logs, training scripts, or inspection wheels cannot leak into the competition archive.


In [ ]:
FILES = {'agent.yaml': 'name: gemma4_swe_agent_v4\n'
               'model: gemma-4-31b-it-qat-w4a16-ct\n'
               'description: Focused Gemma SWE agent that makes an evidence-based production fix and submits '
               'promptly.\n'
               'instruction: !include prompts/system.md\n'
               'tools:\n'
               '  - run_command\n'
               '  - read_file\n'
               '  - edit_file\n'
               '  - submit_patch\n'
               'generate_content_config: !include configs/sampling.yaml\n',
 'configs/sampling.yaml': 'temperature: 0.2\n'
                          'top_p: 0.95\n'
                          'top_k: 40\n'
                          'max_output_tokens: 4096\n'
                          'thinking_config:\n'
                          '  thinking_budget: 2048\n'
                          '  include_thoughts: false\n',
 'eval_config.yaml': 'evaluation:\n'
                     '  timeout_seconds: 120\n'
                     '  max_tool_calls: 16\n'
                     '  max_time_minutes: 6\n'
                     '  max_turns: 12\n',
 'prompts/system.md': 'You are a senior software engineer fixing one issue in `/workspace`. Make a minimal, '
                      'correct production-code patch and submit it.\n'
                      '\n'
                      '## Finish first\n'
                      '- Call `submit_patch()` exactly once, then give a brief text-only completion. If '
                      'prompted again after submission, answer in text and make no tool calls.\n'
                      '- Submit by your 8th tool call at the latest. Submit sooner once the edit and one '
                      'focused review are complete.\n'
                      '- Never call `get_status` or call `submit_patch` a second time.\n'
                      '\n'
                      '## Fast, focused workflow\n'
                      '1. Read the issue carefully. Search once for its exact symbol or distinctive phrase '
                      'using `rg -n -F`. Use the repository layout from `rg --files` only if the named path '
                      'is absent. If a search fails, adjust the path or query once; do not repeat it '
                      'unchanged.\n'
                      '2. Read the smallest useful source range. Do not read the same file/range twice. Use '
                      '`read_file` directly; never type pseudo tool syntax into a shell command.\n'
                      '3. Make a minimal production edit by your 4th tool call. If two source reads do not '
                      'reveal the full design, choose the best evidence-based local fix and submit it '
                      'instead of broadening the search.\n'
                      '4. Tests are read-only. Never edit, create, delete, or rename tests or fixtures. Do '
                      'not create scratch files in the repository. Do not run a test suite; the evaluator '
                      'verifies the patch. At most, run one short check that does not change files.\n'
                      '5. Review the changed lines once, then submit. Do not repeat an identical search, '
                      'read, check, or edit.\n'
                      '\n'
                      'Keep the change small and preserve nearby conventions. Do not access the network, '
                      'install packages, inspect git history, or modify packaging and runner configuration. '
                      'If a tool call fails, correct it once or move on. Never invent test results.\n'}


In [ ]:
import re, zipfile
from pathlib import PurePosixPath

EXPECTED_MODEL = 'gemma-4-31b-it-qat-w4a16-ct'
ALLOWED = {'.yaml', '.yml', '.md', '.txt', '.py', '.json', '.safetensors'}
assert 'agent.yaml' in FILES and len([p for p in FILES if p in {'agent.yaml','agent.yml','root_agent.yaml','root_agent.yml'}]) == 1
for rel, content in FILES.items():
    path = PurePosixPath(rel)
    assert not path.is_absolute() and '..' not in path.parts, rel
    assert path.suffix.lower() in ALLOWED, rel
    if path.suffix.lower() == '.py':
        assert 'skills' in path.parts, rel
    if path.suffix.lower() in {'.yaml', '.yml'}:
        for model in re.findall(r'^\s*model:\s*([^\s#]+)', content, re.M):
            assert model == EXPECTED_MODEL, (rel, model)
        for included in re.findall(r'!include\s+([^\s#]+)', content):
            target = (PurePosixPath(rel).parent / included)
            target = PurePosixPath(*[p for p in target.parts if p not in {'.', ''}])
            assert '..' not in target.parts and target.as_posix() in FILES, (rel, included)

zip_path = WORK / 'submission.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for rel, content in sorted(FILES.items()):
        archive.writestr(rel, content.encode('utf-8'))
with zipfile.ZipFile(zip_path) as archive:
    names = archive.namelist()
    assert 'agent.yaml' in names and not any(n.startswith('../') for n in names)
    assert archive.testzip() is None
    assert set(names) == set(FILES)
print('Created:', zip_path)
print('Archive bytes:', zip_path.stat().st_size)
print('Root agent present:', 'agent.yaml' in names)
print('Files:', len(names))


## Next step

Submit this notebook version's `submission.zip` as the competition artifact. The package uses the competition's required Gemma 4 checkpoint and includes no adapter.
